# Role-Weighted Win Contribution Model
*Which role's gold lead matters most for winning? A per-lane logistic regression on team economic state at different minute interval.*

- **Context fields:** `MATCH_ID`, `AVERAGE_RANK`, `GAME_DATE`, `GAME_DURATION`
- **Features:** `GOLD_DIFF_TOP`, `GOLD_DIFF_JUNGLE`, `GOLD_DIFF_MIDDLE`, `GOLD_DIFF_BOTTOM`, `GOLD_DIFF_SUPPORT` — Blue team's lane-vs-lane-opponent gold differential at minute 15
- **Response:** `BLUE_WIN` — Bernoulli. 1 if Blue team won the match, 0 if Red won

## How to use

1. Run cells top to bottom ('Run' button in the top left)
2. Sections are independently re-runnable after the Data Preparation cell has executed once (`train_df`/`test_df` are held in memory).

## Import Statements

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import DataFrame as SnowflakeDataFrame
from snowflake.snowpark import Session

session = get_active_session()

session.use_warehouse("COMPUTE_WH")
session.use_database("LEAGUE_RECORDS")  
session.use_schema("GOLD")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.model_selection import train_test_split, RepeatedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
)

## Data Preparations
1. Pull Blue-team lane gold diffs (`GOLD.DIFF_INTERVAL_STATE`) at minute 15, joined to match outcome/context (`GOLD.MATCH_TEAM_STATS_SUMMARY`)
2. Exclude remake games (`GAME_DURATION` <= 300)
3. Using Pandas, pivot source data into wide format containing features and response.
4. Preview and diagnose dataset.
5. Train/test split

In [ ]:
def pull_diff_interval_by_match(
    session: Session,
    minute: int = 15,
    team: str = "Blue",
    min_game_duration: int = 300,
) -> SnowflakeDataFrame:
    query = f"""
        SELECT
            D.MATCH_ID,
            D.LANE,
            D.GOLD_DIFF,
            M.WINNING_TEAM,
            M.AVERAGE_RANK,
            M.GAME_DATE,
            M.GAME_DURATION
        FROM GOLD.DIFF_INTERVAL_STATE AS D
        JOIN GOLD.MATCH_TEAM_STATS_SUMMARY AS M
            ON D.MATCH_ID = M.MATCH_ID
        WHERE D.MINUTE = {minute}
          AND D.TEAM = '{team}'
          AND M.GAME_DURATION >= {min_game_duration}
    """
    
    return session.sql(query)

In [ ]:
def pivot_diff_interval(
    df: pd.DataFrame,
    win_reference_team: str = "Blue",
) -> pd.DataFrame:
    lane_rename = {
        "Top": "GOLD_DIFF_TOP",
        "Jungle": "GOLD_DIFF_JUNGLE",
        "Middle": "GOLD_DIFF_MIDDLE",
        "Bottom": "GOLD_DIFF_BOTTOM",
        "Support": "GOLD_DIFF_SUPPORT",
    }
    feature_cols = list(lane_rename.values())
    win_col = f"{win_reference_team.upper()}_WIN"

    pivoted_df = (df
        .pivot_table(
            index=["MATCH_ID", "WINNING_TEAM", "AVERAGE_RANK", "GAME_DATE", "GAME_DURATION"],
            columns="LANE",
            values="GOLD_DIFF",
        )
        .reset_index()
        .rename(columns=lane_rename)
        .assign(**{win_col: lambda d: (d["WINNING_TEAM"] == win_reference_team).astype(int)})
    )
    before = len(pivoted_df)

    pivoted_df = (pivoted_df
        .dropna(subset=feature_cols)
        .reset_index(drop=True)
    )

    print(f"Dropped {before - len(pivoted_df)} matches with incomplete lane data")
    print(f"New pivoted data shape: {pivoted_df.shape}\n")

    return pivoted_df

In [ ]:
def prelim_data(df: pd.DataFrame) -> None:    
    print("--------------- PRELIMINARY DATA VIEW ---------------")
    df.info()
    print("-----------------------------------------------------")
    print(df.describe())
    print("-----------------------------------------------------")
    print(df.head())

In [ ]:
def split_train_test(
    df: pd.DataFrame,
    target_col: str,
    test_size: float = 0.3,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame]:    
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=random_state,
        stratify=df[target_col],
    )
    print(f"Train: {train_df.shape[0]} matches | Test: {test_df.shape[0]} matches")
    print(
        f"Train {target_col} rate: {train_df[target_col].mean():.3f} | "
        f"Test {target_col} rate: {test_df[target_col].mean():.3f}"
    )
    
    return train_df, test_df

### Data Preparations Summary
- **5725 matches** retained after dropping incomplete minute-15 snapshots.
- Class balance: Blue wins **~0.5** of matches — close to 50/50, no resampling needed.
- Split **70/30 train/test**, stratified on `BLUE_WIN`.

## EDA
1. Explore distributions of each lane's `GOLD_DIFF` feature (check normality assumption, skew, outliers).
2. Compute descriptive statistics per feature, split by win/loss.
3. Check feature correlation (multicollinearity risk before fitting).
4. Check class balance of the response variable.

In [ ]:
def plot_feature_distributions(
    df: pd.DataFrame, 
    feature_cols: list[str],
    show_plots: bool = True
) -> pd.DataFrame:
    summary = df[feature_cols].agg(["mean", "std", "skew"]).T

    if show_plots:
        fig, axes = plt.subplots(
            1, 
            len(feature_cols), 
            figsize=(20, 4), 
            sharey=True
        )
        for ax, col in zip(axes, feature_cols):
            sns.histplot(df[col], kde=True, ax=ax)
            ax.set_title(col.replace("GOLD_DIFF_", ""))
            ax.axvline(0, color="red", linestyle="--", linewidth=1)
            
        fig.suptitle("Distribution of Lane Gold Diff at Minute 15")
        plt.tight_layout()
        plt.show()

    return summary

In [ ]:
def plot_feature_correlation(
    df: pd.DataFrame, 
    feature_cols: list[str], 
    show_plots: bool = True
) -> pd.DataFrame:
    """Check feature correlation (multicollinearity risk)."""
    corr = df[feature_cols].corr()

    if show_plots:
        plt.figure(figsize=(6, 5))
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
        plt.title("Lane Gold Diff Correlation")
        
        plt.tight_layout()
        plt.show()

    return corr

In [ ]:
def check_class_balance(
    df: pd.DataFrame, 
    target_col: str, 
    show_plots: bool = True
) -> pd.DataFrame:
    balance = (
        df[target_col]
        .value_counts()
        .rename("count")
        .to_frame()
        .assign(proportion=lambda d: d["count"] / d["count"].sum())
    )

    if show_plots:
        sns.countplot(x=target_col, data=df)
        plt.title(f"{target_col} Class Balance")
        plt.show()

    return balance

### EDA Summary

**Distributions are well-behaved, no transformation needed.**
- All five lane `GOLD_DIFF` features are centered close to zero (means range from −12 to +55, negligible relative to their spread).
- `GOLD_DIFF_SUPPORT` has the tightest spread (std ≈ 847), consistent with supports carrying the least gold overall and having the least room for a lane to diverge economically.
- `GOLD_DIFF_TOP` and `GOLD_DIFF_BOTTOM` have the widest spread (std ≈ 1,759 and 1,897), suggesting these lanes see the largest economic swings by minute 15.

**Correlation is low across most lane pairs, with one expected exception.**
- Most lane pairs correlate weakly (0.08–0.23), well below any multicollinearity concern.
- `GOLD_DIFF_BOTTOM` and `GOLD_DIFF_SUPPORT` correlate at **0.52** — the one notable exception, expected given both players share bot lane and directly affect each other's gold through kill participation and CS/farm splitting.

**Class balance is effectively 50/50, no resampling required.**
- `BLUE_WIN` splits **51.0% / 49.0%**.

## Modelling
1. Scale features to per 1,000g instead of per gold to make interpretation easier.
2. Fit baseline logistic regression on training set.
3. Report initial model diagnostics (coefficients, odds ratios).
4. Predict on test set and compute KPIs (AUC, confusion matrix, classification report).
5. Markdown report of initial fit vs. test performance.

In [ ]:
def scale_gold_diff(df: pd.DataFrame, feature_cols: list[str], scale: float = 1000) -> pd.DataFrame:
    """Rescale features (raw gold units) to per-`scale`-gold units,
    for coefficient legibility. Purely cosmetic: does not change end model KPI."""
    return df.assign(**{
        col: df[col] / scale 
        for col in feature_cols
    })

In [ ]:
def fit_logistic_regression(
    train_df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    **lr_kwargs,
) -> LogisticRegression:
    X_train = train_df[feature_cols]
    y_train = train_df[target_col]

    model = LogisticRegression(max_iter=1000, **lr_kwargs)
    model.fit(X_train, y_train)

    return model

In [ ]:
def report_model_coefficients(
    model: LogisticRegression,
    feature_cols: list[str],
    # -- For refitting with statsmodels for model summary
    train_df: pd.DataFrame = None,
    target_col: str = None,
    show_full_summary: bool = False,
) -> pd.DataFrame:
    """If show_full_summary=True, also fits a statsmodels Logit purely for a
    p-value/CI summary table (train_df and target_col required in that case)."""
    coef_df = (pd
        .DataFrame({
            "feature": feature_cols,
            "coefficient": model.coef_[0],
            "odds_ratio": np.exp(model.coef_[0]),
        })
        .sort_values("coefficient", ascending=False)
        .reset_index(drop=True)
    )
    
    if show_full_summary:
        if train_df is None or target_col is None:
            raise ValueError("train_df and target_col are required when show_full_summary=True")
        
        X_sm = sm.add_constant(train_df[feature_cols])
        y_sm = train_df[target_col]
        sm_model = sm.Logit(y_sm, X_sm).fit(disp=0)
        
        print(sm_model.summary())

    return coef_df

In [ ]:
def evaluate_on_test(
    model: LogisticRegression,
    test_df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    show_report: bool = False,
) -> dict:
    X_test = test_df[feature_cols]
    y_test = test_df[target_col]

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)

    if show_report:
        print(f"Test AUC: {auc:.4f}")
        print("Confusion Matrix:\n", cm)
        print("\nClassification Report:\n", classification_report(y_test, y_pred))

    return {
        "auc": auc,
        "confusion_matrix": cm,
        "classification_report": report,
        "y_proba": y_proba,
    }

In [ ]:
def preview_predictions(
    test_df: pd.DataFrame,
    y_proba: np.ndarray,
    target_col: str,
    n_preview: int = 10,
    show_preview: bool = False,
) -> pd.DataFrame:
    preview_df = (test_df[["MATCH_ID", target_col]]
        .rename(columns={target_col: "actual"})
        .assign(predicted_proba=y_proba)
        .assign(predicted=(y_proba >= 0.5).astype(int))
        .assign(correct=lambda df: df["actual"] == df["predicted"])
    )

    if show_preview:
        print(preview_df.head(n_preview))

    return preview_df

In [ ]:
def plot_roc_curve(
    y_test: pd.Series, 
    y_proba: np.ndarray, 
    show_plots: bool = True
) -> pd.DataFrame:
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    roc_df = pd.DataFrame({
        "fpr": fpr, 
        "tpr": tpr, 
        "threshold": thresholds
    })

    if show_plots:
        auc = roc_auc_score(y_test, y_proba)
        plt.figure(figsize=(5, 5))
        plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
        plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
        
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve")
        plt.legend()
        
        plt.tight_layout()
        plt.show()

    return roc_df

### Modelling — initial fit summary

**All five lane coefficients are positive and statistically significant (p < 0.01), confirming a gold lead in any lane raises Blue's win probability.**
- Jungle and Middle carry the largest coefficients (~0.48 per 1,000g), Bottom close behind (0.42).
- Top (0.28) and especially Support (0.17) matter meaningfully less — consistent with earlier exploratory findings.
- Intercept is not significant (p = 0.459), as expected — an even game should be ~50/50 with no inherent side bias.

**Test AUC of 0.83; accuracy sits lower at 74%.**
- Precision/recall are balanced across both classes (~0.73–0.76).
- Note that rerunning the same model on later intervals improve AUC, but also has diminishing returns. Expected, as most games are decided between 15-25 minutes.

**Errors cluster in genuinely close games (0.40–0.58 predicted), confident predictions (0.89+) are correct.**

Overall, baseline model has good performance KPI with a simple, interpretable spec.

## Cross-Validation and Coefs
1. Refit logistic regression across repeated k-fold resamples (k=10, 5 repeats) to build an empirical sampling distribution of each lane's coefficient.
2. Compute mean, std, and 95% CI per coefficient.
3. Build final per-lane importance table and bar chart.
4. Markdown report comparing coefficient stability against the initial single fit.

In [ ]:
def coefficient_stability_cv(
    # -- Base Inputs
    train_df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    # -- Cross-validation Parameters
    n_splits: int = 10,
    n_repeats: int = 5,
    random_state: int = 42,
    show_summary: bool = False,
) -> pd.DataFrame:
    """Refit logistic regression across repeated k-folds for sampling distributions of coefs."""
    rkf = RepeatedKFold(
        n_splits=n_splits, 
        n_repeats=n_repeats, 
        random_state=random_state
    )
    
    fold_coefs = []
    X = train_df[feature_cols]
    y = train_df[target_col]

    for train_idx, _ in rkf.split(X):
        X_fold, y_fold = X.iloc[train_idx], y.iloc[train_idx]
        model = LogisticRegression(max_iter=1000).fit(X_fold, y_fold)
        fold_coefs.append(model.coef_[0])

    fold_coefs = np.array(fold_coefs)

    summary = (pd
        .DataFrame({
            "feature": feature_cols,
            "mean_coef": fold_coefs.mean(axis=0),
            "std_coef": fold_coefs.std(axis=0, ddof=1),
        })
        .assign(ci_lower_95=lambda df: df["mean_coef"] - 1.96 * df["std_coef"])
        .assign(ci_upper_95=lambda df: df["mean_coef"] + 1.96 * df["std_coef"])
        .assign(ci_lower_95_pct=np.percentile(fold_coefs, 2.5, axis=0))
        .assign(ci_upper_95_pct=np.percentile(fold_coefs, 97.5, axis=0))
        .sort_values("mean_coef", ascending=False)
        .reset_index(drop=True)
    )

    if show_summary:
        print(summary)

    return summary

In [ ]:
def plot_lane_importance(
    stability_summary: pd.DataFrame,
    show_plots: bool = True,
    decimals: int = 4,
) -> pd.DataFrame:
    """Final section: build and plot the per-lane coefficient summary as a
    ranked bar chart. Coefficients are transformed into odds ratios
    (exp(coef)) for stakeholder legibility -- exact everywhere, unlike a
    marginal-probability transform which only holds near an even game.
    Raw log-odds coefficients are retained in the returned dataframe for
    technical reference."""
    lane_importance = (stability_summary
        .assign(lane=lambda df: df["feature"].str.replace("GOLD_DIFF_", ""))
        .assign(odds_ratio=lambda df: np.exp(df["mean_coef"]))
        .assign(odds_ratio_lower=lambda df: np.exp(df["ci_lower_95"]))
        .assign(odds_ratio_upper=lambda df: np.exp(df["ci_upper_95"]))
        .assign(odds_ratio_err=lambda df: (df["odds_ratio_upper"] - df["odds_ratio_lower"]) / 2)
    )

    if show_plots:
        plt.figure(figsize=(10, 6))
        plt.barh(
            lane_importance["lane"],
            lane_importance["odds_ratio"],
            xerr=lane_importance["odds_ratio_err"],
            capsize=4,
            color="#5B8DEF",
        )
        
        plt.axvline(1.0, color="gray", linestyle="--", linewidth=1)
        plt.xlabel("Odds ratio per 1,000 gold lead")
        plt.ylabel("Lane")
        plt.title("Lane Importance: Odds Multiplier per 1,000g Lead\n(mean ± 95% CI, 10x5 CV)")
        plt.gca().invert_yaxis()
        
        plt.tight_layout()
        plt.show()

    output_cols = [
        "lane", "mean_coef", "std_coef", "ci_lower_95", "ci_upper_95",
        "odds_ratio", "odds_ratio_lower", "odds_ratio_upper",
    ]
    result = lane_importance[output_cols].round(decimals)
    return result

## Run and Report

In [ ]:
def role_importance(
    session: Session,
    minute: int = 15,
    team: str = "Blue",
    test_size: float = 0.3,
    random_state: int = 42,
    gold_scale: float = 1000,
    result_decimals: int = 4,
    # -- Show visuals and prints
    show_prelim: bool = False,
    show_eda_plots: bool = False,
    show_model_summary: bool = False,
    show_test_report: bool = False,
) -> dict:
    """Notebook total orchestration call: 
    Data Preparation > EDA > Modelling > Cross-Validation and Report. 
    Returns a bundle of all artifacts a downstream consumer (e.g. the Streamlit app) 
    might need."""
    target_col = f"{team.upper()}_WIN"
    feature_cols = [
        "GOLD_DIFF_TOP", "GOLD_DIFF_JUNGLE", "GOLD_DIFF_MIDDLE",
        "GOLD_DIFF_BOTTOM", "GOLD_DIFF_SUPPORT",
    ]

    # ----- Data Preparation
    source_data = pull_diff_interval_by_match(session, minute=minute, team=team)
    df = source_data.to_pandas()
    pivoted_df = pivot_diff_interval(df, win_reference_team=team)
    if show_prelim:
        prelim_data(pivoted_df)
        
    train_df, test_df = split_train_test(
        pivoted_df, 
        target_col=target_col, 
        test_size=test_size, 
        random_state=random_state
    )

    # ----- EDA
    dist_summary = plot_feature_distributions(train_df, feature_cols, show_plots=show_eda_plots)
    corr_df = plot_feature_correlation(train_df, feature_cols, show_plots=show_eda_plots)
    balance_df = check_class_balance(train_df, target_col, show_plots=show_eda_plots)

    # ----- Modelling
    train_scaled = scale_gold_diff(train_df, feature_cols, scale=gold_scale)
    test_scaled = scale_gold_diff(test_df, feature_cols, scale=gold_scale)

    baseline_model = fit_logistic_regression(train_scaled, feature_cols, target_col)
    coef_df = report_model_coefficients(
        baseline_model, feature_cols,
        train_df=train_scaled, target_col=target_col,
        show_full_summary=show_model_summary,
    )
    test_metrics = evaluate_on_test(
        baseline_model, test_scaled, feature_cols, target_col, show_report=show_test_report
    )
    prediction_preview = preview_predictions(
        test_scaled, test_metrics["y_proba"], target_col,
        show_preview=show_test_report,
    )
    roc_df = plot_roc_curve(
        test_scaled[target_col], test_metrics["y_proba"], show_plots=show_eda_plots
    )

    # ----- Cross-Validation and Report
    stability_summary = coefficient_stability_cv(
        train_scaled, feature_cols, target_col,
        n_splits=10, n_repeats=5,
        show_summary=show_test_report,
    )
    lane_importance = plot_lane_importance(stability_summary, show_plots=show_eda_plots)

    return {
        "model": baseline_model,
        "feature_cols": feature_cols,
        "target_col": target_col,
        "gold_scale": gold_scale,
        "lane_importance": lane_importance,
        "test_metrics": test_metrics,
        "coef_df": coef_df,
        "train_df": train_df,
        "test_df": test_df,
    }

In [ ]:
role_importance(
    session, 
    minute=15, 
    team='Blue', 
    show_eda_plots=True
)

# Final Summary

**The coefficient-stability check confirms the initial fit — nothing shifted after resampling.**
- Mean coefficients from 10x5 repeated CV are nearly identical to the single baseline fit (e.g. Middle 0.4786 vs 0.4786, Jungle 0.4764 vs 0.4764) — the original split wasn't a lucky draw.
- Ranking is stable across all 50 resamples: **Middle ≈ Jungle > Bottom > Top > Support**, every time, no exceptions.

**Middle and Jungle are statistically tied for most important; the other three lanes are clearly separated.**
| Lane | Odds Ratio | 95% CI | Note |
|---|---|---|---|
| Middle | 1.61 | [1.59, 1.64] | Tied with Jungle — CIs overlap, data can't distinguish which is truly larger |
| Jungle | 1.61 | [1.58, 1.64] | Tied with Middle — CIs overlap, data can't distinguish which is truly larger |
| Bottom | 1.52 | [1.49, 1.56] | Non-overlapping with neighbors — ranking is solid |
| Top | 1.33 | [1.31, 1.35] | Non-overlapping with neighbors — ranking is solid |
| Support | 1.19 | [1.14, 1.24] | Non-overlapping with neighbors — ranking is solid |

**In plain terms:** every 1,000 gold Blue leads by in Mid or Jungle roughly multiplies their odds of winning by 1.6x. The same lead in Support only multiplies odds by 1.2x — a support player being ahead of their lane opponent economically barely moves the needle on who wins, while a mid or jungle lead moves it substantially.

**Performance holds at 0.83 AUC / 74% accuracy**, matching the initial fit exactly — confirms the coefficient estimates and the model's predictive behavior are consistent, not an artifact of the original 70/30 split.

**Bottom line:** this model gives a defensible, resampling-validated answer to the core question — Mid and Jungle leads matter most for winning, Support leads matter least. Continuously ingest more data and the data will automatically update itself.